# Zenteno S · calibración con temperatura offline y causal

Se ajusta el modelo S del notebook 17 sobre las mismas muestras LAB004–009 en dos condiciones. El ajuste offline usa el perfil reconstruido de la calibración original. El ajuste causal usa la mediana del último minuto cerrado. Ambos usan σX = 0,50 g/L, los mismos cinco parámetros libres, cotas, semillas, pulsos y malla RK4.

Para evaluar ambos vectores se usa la misma temperatura causal en LAB004–009 y LAB016–024. Se comparan la química, el RMSE del flujo de CO₂ y la altura y hora de su pico por lote. Los coeficientes de gas proceden del modelo S del notebook 17 y se usan en ambas evaluaciones.

## Solubilidad variable y predicción de gas
Se usan los coeficientes de transferencia y ganancia calibrados en el notebook 17. Cambia la trayectoria química que alimenta el cálculo de gas.

The empirical wine correlation reproduced as Eq. 1 in Liger-Belair et al. (2023), attributed there to Lonvaud-Funel and Matsumoto, is
\[
k_H=3.295\exp[(7\times10^{-5}a-0.02905)T-0.0179a-0.00111S],\qquad C_{sat}=k_Hp_{CO_2}.
\]
T is in °C, a is ethanol % vol, S is total sugar in g/L, kH is in g/(L bar). Ethanol conversion uses a=E/7.8924, with pure ethanol density 789.24 g/L at 20°C as a reference conversion, not an evolving mixture density. Measured temperature and model-predicted ethanol/sugar vary throughout each batch. Coefficients are fixed; pCO2=1 bar is an explicit assumption, not a pressure measurement. Application to fermenting must is an extrapolation requiring validation, particularly at high sugar. Source: https://doi.org/10.1021/acsomega.3c01812 . The historical provisional correlation is not used.

The transfer law below is our adaptation, not an equation attributed to that paper:
\[
q_{eff}=Aq_{prod},\quad r_{abs}=\min(q_{eff},k_La\max(C_{sat}-C_d,0)),\quad
r_{des}=k_La\max(C_d-C_{sat},0),\quad
\dot C_d=r_{abs}-r_{des},\quad q_{gas}=q_{eff}-\dot C_d,\quad q_{pred}=gq_{gas}.
\]
This retains the previous absorption-limited reservoir and adds release above equilibrium. Cd is not clipped to changing saturation: supersaturation can persist and relax. Cd(0)=0 is retained. The same effective source activation is retained, so conservation applies to that source, not to unattenuated stoichiometric production. kLa is an effective coefficient in this adapted law, not an independently measured physical transfer coefficient.

Exact piecewise integration for interval-constant midpoint saturation and interval-mean production preserves positivity and concentration balance. Finite-volume mass checks include homogeneous withdrawals; gain is outside the balance.
Los intervalos de gas se dividen en cada actualización del sensor. La temperatura disponible al inicio del intervalo se mantiene al calcular la producción y la solubilidad dentro de él.

In [ ]:
from pathlib import Path
import re
from dataclasses import replace
import sys,json,hashlib,math
import nbformat
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/wine_fermentation').is_dir())
try:
    from numba import njit
except ImportError:
    def njit(function):
        return function
sys.path.insert(0,str(ROOT/'src'));sys.dont_write_bytecode=True
from wine_fermentation import model as kin
from wine_fermentation.data import natural_batches
from wine_fermentation.models.historical.calibration import load
from wine_fermentation.comparison import observations,chemical_observations,CHEMICAL_COLUMNS
protected=[p for folder in ['src','configs','data/compatibility/chemistry','data/cleaned/natural/lab004_009'] for p in (ROOT/folder).rglob('*') if p.is_file() and '__pycache__' not in str(p) and '_variable_solubility.' not in p.name]
protected += [ROOT/'notebooks/16_zenteno_recalibration_legacy_treatment_variable_solubility.source.ipynb']
hashes={p:hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}
data,training,_=natural_batches(ROOT)
# Correcciones de recuento inicial indicadas por el usuario; originales conservados.
INITIAL_VIABLE_MCELLS_ML={'LAB004':10.75,'LAB005':11.25}
initial_biomass_audit=[];corrected_training=[]
for batch in training:
 if batch.batch in INITIAL_VIABLE_MCELLS_ML:
  count=INITIAL_VIABLE_MCELLS_ML[batch.batch];x0=count*.03
  ids=np.flatnonzero(np.isclose(batch.time,0.));assert len(ids)==1
  obs0={k:v.copy() for k,v in batch.observations.items()};obs0['X'][ids[0]]=x0
  initial_biomass_audit.append(dict(LAB=batch.batch,original_g_L=batch.initials['X'],viables_millones_mL=count,corregido_g_L=x0))
  batch=replace(batch,initials={**batch.initials,'X':x0},observations=obs0)
  mask=data.batch.eq(batch.batch)&np.isclose(data.time_h,0.)
  assert mask.sum()==1
  data.loc[mask,'X_viable_kg_m3']=x0;data.loc[mask,'X_viable_mcells_ml']=count
  assert batch.initials['X']==batch.observations['X'][ids[0]]
 corrected_training.append(batch)
training=corrected_training
assert len(initial_biomass_audit)==2
display(pd.DataFrame(initial_biomass_audit))

gf_seed=kin.normalize_parameters(json.loads((ROOT/'configs/theta/theta_full.json').read_text()))
published_theta={'mu0': 0.09085058816701197, 'betaS0': 0.8189664538991809, 'Ks0': 23.046564567082594, 'Kie0': 96.26516563486751, 'Yxn': 6.497331613020979, 'Yes': 0.4893180893478443}
fixed=dict(Kie0=published_theta['Kie0'],Kn0=.008257860224319971,Kd0=.00044,
           m0=gf_seed['m0'],Yxs=1/(1/gf_seed['Yxg']+1/gf_seed['Yxf']))
published_theta.update(fixed)
start={**fixed,'mu0':.09403147367985473,'betaS0':.42243676903335664+.6236552480900203,
       'Ks0':10.,'Yxn':6.456225797038313,'Yes':.49}
bounds=dict(mu0=(.05,1.),betaS0=(.1,20.),Ks0=(.1,200.),Yxn=(1.,100.),Yes=(.1,.5114))
names=list(bounds)
lo,hi=np.array(list(bounds.values())).T
STATES=('X','N','S','E')
S_PARAMETERS=(*names,*fixed)
display(pd.Series(fixed,name='Fijos del modelo S del notebook 17'))
display(pd.DataFrame([dict(parámetro=k,inicio=start[k],mínimo=lo[i],máximo=hi[i]) for i,k in enumerate(names)]))

## Observaciones y ponderación

Se usan todas las muestras válidas posteriores a la inicial y σX = 0,50 g/L. Para el YAN se aplica el tratamiento del notebook 16, incluido el agotamiento a las 122 h de LAB006. Para S se suman G y F de la misma muestra y sus escalas de error se combinan en cuadratura.

El perfil offline es el de la calibración original, construido por el lector común a partir de medianas de 30 minutos, suavizado e interpolación. El perfil causal toma la última mediana disponible de un minuto cerrado y la mantiene durante los huecos. En ambos ajustes el estado inicial se mantiene hasta la primera temperatura causal disponible. La malla, los saltos de N y el paso máximo de un minuto son comunes. Para cada intervalo se toma el valor del perfil correspondiente en su extremo izquierdo y se usa en todas las etapas de RK4.

In [ ]:
treated=[];audit=[];rules=[]
for b in training:
 obs={k:v.copy() for k,v in b.observations.items()};old=obs['N'].copy()
 obs['N'][np.isfinite(old)&(old<=.020)]=0
 if b.batch=='LAB006':
  ids=np.flatnonzero(np.isclose(b.time,122)&np.isfinite(old));assert len(ids)==1 and np.isclose(old[ids[0]],.027)
  obs['N'][ids]=0;depletion=122.
 else:
  last=max(t for t,d in b.pulses['N']);ids=np.flatnonzero((b.time>=last)&np.isfinite(obs['N'])&(obs['N']==0));assert len(ids)
  depletion=float(b.time[ids[0]])
 obs['N'][b.time>depletion+12]=np.nan
 for i in np.flatnonzero(np.isfinite(old)):
  audit.append(dict(LAB=b.batch,t_h=b.time[i],original_mg=old[i]*1000,usado_mg=obs['N'][i]*1000,estado='Inicial' if b.time[i]==0 else ('Excluido' if not np.isfinite(obs['N'][i]) else ('Transformado' if old[i]!=obs['N'][i] else 'Usado'))))
 assert all(np.array_equal(obs[k],b.observations[k],equal_nan=True) for k in obs if k!='N')
 treated.append(replace(b,observations=obs))
 rules.append(dict(LAB=b.batch,agotamiento_h=depletion,corte_YAN_h=depletion+12,N0_mg=b.initials['N']*1000,pulsos=b.pulses['N'],**{k:int(np.count_nonzero(np.isfinite(obs[k])&(b.time>0))) for k in obs}))
audit=pd.DataFrame(audit);display(pd.DataFrame(rules));display(audit)
def sigma(state,y,sigma_x):return np.full_like(y,sigma_x) if state=='X' else kin.sigma_for_state(state,y)
thermal_profiles={}

def register_temperature(lab, raw):
    sensor=raw[['t_h','T']].apply(pd.to_numeric,errors='coerce').dropna()
    sensor=sensor.loc[np.isfinite(sensor).all(axis=1)].sort_values('t_h')
    minute=np.floor(sensor.t_h.to_numpy()*60).astype(int)
    medians=sensor.groupby(minute)['T'].median()
    if medians.empty:
        raise ValueError(f'{lab}: registro de temperatura vacío')
    available=(medians.index.to_numpy()+1)/60
    assert np.all(np.diff(available)>0)
    # Una lectura contribuye únicamente después del cierre de su minuto.
    assert np.all(sensor.t_h.to_numpy()<(minute+1)/60+1e-12)
    thermal_profiles[lab]=(available,medians.to_numpy(float))


def causal_temperature(batch,time):
    available,values=thermal_profiles[batch.batch]
    index=np.searchsorted(available,time+1e-12,side='right')-1
    if np.any(index<0):
        raise ValueError(f'{batch.batch}: temperatura aún no disponible')
    return values[index]


def propagation_start(batch):
    return max(float(batch.time[0]),float(thermal_profiles[batch.batch][0][0]))


clocks=pd.read_csv(ROOT/'data/compatibility/chemistry/clocks.csv').set_index('batch').t0
temp_paths=list((ROOT/'data/original/natural/process_logs').rglob('*.csv'))
for batch in treated:
    files=sorted(path for path in temp_paths if path.name.lower().startswith('temp') and
                 re.search(rf'lab0*{int(batch.batch[3:])}(?:\D|$)',path.name,re.I))
    raw=pd.concat([pd.read_csv(path,usecols=['timestamp','T']) for path in files])
    raw['timestamp']=pd.to_datetime(raw.timestamp,errors='coerce')
    raw['T']=pd.to_numeric(raw['T'],errors='coerce')
    assert raw.groupby('timestamp')['T'].nunique().le(1).all()
    raw=raw.dropna().drop_duplicates('timestamp')
    hours=(raw.timestamp-pd.Timestamp(clocks.loc[batch.batch])).dt.total_seconds()/3600
    register_temperature(batch.batch,raw.assign(t_h=hours))


R_GAS=kin.FIXED_CONSTANTS['R']
MAINTENANCE_CUTOFF=kin.MAINTENANCE_SUGAR_CUTOFF_KG_M3

@njit
def sugar_terms(y,p,temperature_c):
    _,n,s,e=np.maximum(y,0.)
    mu0,beta_s0,ks0,_,_,kie0,kn0,kd0,m0,_=p
    tk=temperature_c+273.15
    eps=1e-8
    a_k=math.exp(46055.*(tk-293.15)/(293.15*R_GAS*tk))
    a_mu=math.exp(59453.*(tk-300.)/(300.*R_GAS*tk))
    a_beta=math.exp(11000.*(tk-296.15)/(296.15*R_GAS*tk))
    kie=kie0*(a_k+eps)
    mu=mu0*a_mu*n/(n+kn0*a_k+eps) if s>0 else 0.
    beta=beta_s0*a_beta*s/(s+ks0*a_k+eps)*kie/(e+kie)
    threshold=-.0001*e**3+.0049*e**2-.1279*e+315.89
    kd=kd0*math.exp(.0415*e+130000.*(tk-305.65)/(305.65*R_GAS*tk)) if tk>=threshold else 0.
    maintenance=m0*math.exp(37681.*(tk-293.3)/(293.3*R_GAS*tk))*s/(s+MAINTENANCE_CUTOFF)
    return mu,beta,kd,maintenance


@njit
def chemical_rhs(y,p,temperature_c):
    x=max(y[0],0.)
    mu,beta,kd,maintenance=sugar_terms(y,p,temperature_c)
    yxn,yes,yxs=p[3],p[4],p[-1]
    return x*np.array([mu-kd,-mu/yxn,-(mu/yxs+beta/yes+maintenance),beta])


@njit
def integrate_chemical(initial,step_sizes,temperatures,doses,p):
    state=initial.copy()
    values=np.empty((len(doses),len(initial)))
    values[0]=initial
    for i,dt in enumerate(step_sizes):
        temperature=temperatures[i]
        k1=chemical_rhs(state,p,temperature)
        k2=chemical_rhs(state+dt*k1/2,p,temperature)
        k3=chemical_rhs(state+dt*k2/2,p,temperature)
        k4=chemical_rhs(state+dt*k3,p,temperature)
        state=np.maximum(state+dt*(k1+2*k2+2*k3+k4)/6,0.)
        values[i+1]=state
        state[1]+=doses[i+1]
    return values


def parameter_vector(parameters):
    return np.array([parameters[key] for key in S_PARAMETERS])


chemical_meshes={}

def chemical_mesh(batch,max_step,temperature_mode):
    key=(batch.batch,max_step,temperature_mode)
    if key in chemical_meshes:
        return chemical_meshes[key]
    times=np.asarray(batch.time)
    begin=propagation_start(batch)
    end=times[-1]
    available,_=thermal_profiles[batch.batch]
    events={}
    for time,dose in batch.pulses['N']:
        assert begin<time and dose>=0
        if time<=end:
            events[time]=events.get(time,0.)+dose
    mesh=np.unique(np.r_[times,begin,list(events),np.arange(begin,end,max_step),
                         available[(available>begin)&(available<end)]])
    doses=np.zeros(len(mesh))
    pulse_indices=np.searchsorted(mesh,list(events))
    doses[pulse_indices]=list(events.values())
    # Ambos perfiles se muestrean en las mismas horas.
    active=mesh[:-1]>=begin
    interval_starts=np.maximum(mesh[:-1],begin)
    if temperature_mode=='offline':
        temperatures=np.interp(interval_starts,batch.time,batch.temperature_c)
    else:
        assert temperature_mode=='causal'
        temperatures=causal_temperature(batch,interval_starts)
    steps=np.where(active,np.diff(mesh),0.)
    rows=np.searchsorted(mesh,times)
    assert np.array_equal(mesh[rows],times)
    chemical_meshes[key]=(steps,temperatures,doses,rows,pulse_indices,mesh)
    return chemical_meshes[key]


def simulate(batch,parameters,max_step=1/60,*,temperature_mode='causal'):
    steps,temperatures,doses,rows,pulse_indices,mesh=chemical_mesh(batch,max_step,temperature_mode)
    initial=batch.initials
    y0=np.array([initial['X'],initial['N'],initial['G']+initial['F'],initial['E']])
    states=integrate_chemical(y0,steps,temperatures,doses,parameter_vector(parameters))
    if not np.isfinite(states).all():
        raise RuntimeError(f'{batch.batch}: integración química inválida')
    frame=pd.DataFrame(states[rows],index=batch.time,columns=STATES)
    jumps=[]
    for index in pulse_indices:
        before=states[index].copy()
        after=before.copy()
        after[1]+=doses[index]
        jumps.append(dict(t=mesh[index],dose=doses[index],before=before,after=after))
    frame.attrs['pulse_jumps']=jumps
    frame.attrs['minimum_integrated_states']=dict(zip(STATES,states.min(axis=0)))
    return frame


def with_jumps(f):
 extra=[];tt=[]
 for j in f.attrs['pulse_jumps']:
  if j['t'] not in f.index:extra.append(j['before']);tt.append(j['t'])
  extra.append(j['after']);tt.append(j['t'])
 if not extra:return f
 added=pd.DataFrame(extra,index=tt,columns=STATES)
 return pd.concat([f,added]).sort_index(kind='stable')


def observed_state(batch,state,sigma_x):
    if state=='S':
        glucose,fructose=(batch.observations[key] for key in ('G','F'))
        scale=np.hypot(kin.sigma_for_state('G',glucose),kin.sigma_for_state('F',fructose))
        return glucose+fructose,scale
    values=batch.observations[state]
    return values,sigma(state,values,sigma_x)

## Recalibración química pareada

La tabla de convergencia puntúa cada ajuste con su perfil de temperatura de calibración. Las figuras químicas y gaseosas siguientes simulan ambos con temperatura causal.

In [ ]:
def fit_chemistry(observations, sigma_x, temperature_mode):
 support=[]
 for batch in observations:
  for state in STATES:
   values,scale=observed_state(batch,state,sigma_x)
   mask=np.isfinite(values)&(batch.time>0)
   support.append((batch,state,values[mask],scale[mask],mask))
 NR=sum(len(values) for _,_,values,_,_ in support)
 calls = rejected = 0
 def residual(logp):
  nonlocal calls,rejected
  calls+=1;p={**start,**dict(zip(names,np.exp(logp)))}
  try:pred={b.batch:simulate(b,p,temperature_mode=temperature_mode) for b in observations}
  except (RuntimeError,ValueError,FloatingPointError):
   rejected+=1;return np.full(NR,1e4)
  e=np.concatenate([(pred[b.batch][s].to_numpy()[mask]-values)/scale for b,s,values,scale,mask in support])
  if calls%50==0:print('Evaluaciones',calls,'MSE',float(np.mean(e*e)),'rechazos',rejected,flush=True)
  return e
 rng=np.random.default_rng(20261101);x0=np.array([start[k] for k in names]);seeds=[x0,x0*np.exp(rng.normal(0,.2,len(names)))];runs=[];runrows=[]
 for j,seed in enumerate(seeds):
  sol=least_squares(residual,np.log(np.clip(seed,lo+1e-10,hi-1e-10)),bounds=(np.log(lo),np.log(hi)),max_nfev=100,diff_step=1e-3,ftol=2e-5,xtol=2e-5,gtol=2e-5)
  runs.append(sol);runrows.append(dict(inicio=j,success=sol.success,nfev=sol.nfev,MSE=float(np.mean(sol.fun**2)),message=sol.message));print(runrows[-1],flush=True)
 best=min((run for run in runs if run.success),key=lambda s:s.fun@s.fun);theta={**start,**dict(zip(names,np.exp(best.x)))}
 assert theta['Yes']<=.5114 and all(theta[k]==v for k,v in fixed.items())
 return theta, pd.DataFrame(runrows), NR, rejected


offline_label = 'Ajuste offline'
causal_label = 'Ajuste causal'
SIGMA_X = .50
parameters = {}

for batch in treated:
    offline = chemical_mesh(batch,1/60,'offline')
    causal = chemical_mesh(batch,1/60,'causal')
    for field in (0,2,3,4,5):
        np.testing.assert_array_equal(offline[field],causal[field])
assert any(not np.array_equal(chemical_mesh(b,1/60,'offline')[1],
                              chemical_mesh(b,1/60,'causal')[1]) for b in treated)

for label, temperature_mode in ((offline_label,'offline'),(causal_label,'causal')):
    print(label, flush=True)
    parameters[label], runs, n, rejected = fit_chemistry(treated, SIGMA_X, temperature_mode)
    display(runs.assign(ajuste=label, sigma_X_g_L=SIGMA_X, n_residuos=n, rechazos=rejected))

theta = parameters[causal_label]
parameter_table = pd.DataFrame(parameters).assign(libre=lambda f: f.index.isin(names))
display(parameter_table)
print('TEMPERATURE_FIT_PARAMETERS_JSON', json.dumps(parameters))
traincurves = {b.batch: simulate(b, theta) for b in treated}
reference_train = {offline_label: {b.batch: simulate(b,parameters[offline_label]) for b in treated}}

## Perfiles durante la calibración · LAB004–009

Las líneas verticales marcan el inicio común de la integración. Los dos ajustes se evalúan después con la entrada causal.

In [ ]:
def temperature_panel(ax,batch,role=''):
 available,values=thermal_profiles[batch.batch]
 first,end=batch.time[[0,-1]]
 mask=(available>=first)&(available<=end)
 ax.scatter(available[mask],values[mask],s=2,c='black',alpha=.35,label='Completed-minute medians')
 start=propagation_start(batch)
 knots=np.r_[start,available[(available>start)&(available<end)],end]
 ax.step(knots,causal_temperature(batch,knots),where='post',c='#0072B2',lw=1,
         label='Causal model input')
 ax.set(title=batch.batch+role,xlabel='Time [h]',ylabel='Temperature [°C]')
 ax.grid(alpha=.2)
 ax.legend(loc='best',fontsize=8)

fig,axes=plt.subplots(2,3,figsize=(15,8))
for ax,batch in zip(axes.flat,treated):
 temperature_panel(ax,batch)
 begin=propagation_start(batch)
 knot=np.unique(np.r_[begin,batch.time[batch.time>=begin]])
 ax.plot(knot,np.interp(knot,batch.time,batch.temperature_c),c='#D55E00',lw=1.2,
         linestyle='--',label='Offline model input')
 ax.axvline(begin,c='gray',ls=':',lw=.8,label='Common start')
 ax.legend(loc='best',fontsize=8)
fig.suptitle('Temperature · Calibration LAB004–009',y=.99)
fig.tight_layout(pad=.7,h_pad=1.2,w_pad=.9)
plt.show()

## Evaluación con temperatura causal · LAB004–009

Los lotes de esta sección también se usaron para ajustar los parámetros químicos. El RMSE se calcula sobre las mismas muestras posteriores a la inicial en ambas curvas.

In [ ]:
def legend_inside(ax):
 handles,labels=ax.get_legend_handles_labels()
 unique=dict(zip(labels,handles))
 legend=ax.legend(unique.values(),unique.keys(),loc='best',fontsize=8,framealpha=.85,
                  title=getattr(ax,'_rmse_text',None),title_fontsize=8)
 assert len(legend.get_texts())==len(unique)


styles = {offline_label: dict(color='#D55E00', linestyle='--'),
          causal_label: dict(color='#0072B2', linestyle='-')}
paired_scores = []
biomass_errors = []


def plot_comparison(ax, traces, horizon):
 for label, trace in traces.items():
  window = trace.index <= horizon
  ax.plot(trace.index[window], trace[window], label=label, **styles[label])


def score_comparison(group, lab, state, unit, times, observed, predictions):
 if not len(times):
  return None
 rmse = {}
 for label, predicted in predictions.items():
  assert len(predicted)==len(times) and np.isfinite(predicted).all() and np.isfinite(observed).all()
  errors = predicted - observed
  rmse[label] = np.sqrt(np.mean(errors**2))
  paired_scores.append(dict(grupo=group, LAB=lab, variable=state, unidad=unit,
                            ajuste=label, n=len(times), RMSE=rmse[label], sesgo=errors.mean()))
  if state=='X' and group!='Calibración química':
   for time, sample, value in zip(times, observed, predicted):
    biomass_errors.append(dict(LAB=lab,t_h=time,ajuste=label,observada=sample,
                               predicha=value,error=value-sample))
 return (f'RMSE ({unit}) · offline {rmse[offline_label]:.3f}\n'
         f'causal {rmse[causal_label]:.3f}')


def chemical_frames(lab, fitted, references):
 return {**{label: frames[lab] for label,frames in references.items()}, causal_label: fitted[lab]}


for state,title,scale in [('X','Biomass',1),('N','YAN',1000),('S','Total sugar',1),('E','Ethanol',1)]:
 fig,axes=plt.subplots(2,3,figsize=(15,8))
 unit='mg/L' if state=='N' else 'g/L'
 for ax,batch in zip(axes.flat,treated):
  frames=chemical_frames(batch.batch,traincurves,reference_train)
  plot_comparison(ax, {label: with_jumps(frame)[state]*scale for label,frame in frames.items()}, batch.time[-1])
  measured,_=observed_state(batch,state,SIGMA_X)
  used=np.isfinite(measured)&(batch.time>0)
  ax.scatter(batch.time[used],measured[used]*scale,c='black',s=25,label='Samples')
  initial=batch.initials
  initial_value=initial['G']+initial['F'] if state=='S' else initial[state]
  ax.scatter([0.],[initial_value*scale],c='black',s=25,label='Samples')
  predictions={label: frame[state].to_numpy()[used]*scale for label,frame in frames.items()}
  ax._rmse_text=score_comparison('Calibración química',batch.batch,state,unit,batch.time[used],
                                 measured[used]*scale,predictions)
  if state=='N':
   for time,dose in batch.pulses['N']:
    ax.axvline(time,c='gray',ls=':',lw=.8,label='N pulse')
  ax.set(title=batch.batch,xlabel='Time [h]',ylabel=f'{title} [{unit}]',ylim=(0,None))
  ax.grid(alpha=.2)
  legend_inside(ax)
 fig.suptitle(title+' · Calibration LAB004–009',y=.99)
 fig.tight_layout(pad=.7,h_pad=1.2,w_pad=.9)
 plt.show()

## Predicciones LAB016–021

LAB019–021 se usaron en el notebook 17 para calibrar el gas del modelo S. La biomasa de LAB016–021 se reserva para evaluar los dos ajustes químicos.

In [ ]:
v=load(ROOT);LABS=['LAB019','LAB020','LAB021','LAB016','LAB017','LAB018'];TRAIN=LABS[:3];HOLD=LABS[3:]
raw=pd.read_csv(ROOT/'data/processed/co2_mass_balance_lab016_021_g_l_h.csv',usecols=['lab','time_h','record_kind','remaining_volume_L','co2_raw_g_l_h']);obs=observations(raw)
batches={};cores={};volumes={};caches={};activation={};entry=[]
for lab in LABS:
 b=v.batches[lab];ev=raw.loc[raw.lab.eq(lab)&raw.record_kind.eq('withdrawal_after')].sort_values('time_h');et=ev.time_h.to_numpy();vv=ev.remaining_volume_L.to_numpy()
 def volume(t,et=et,vv=vv):return np.r_[2.,vv][np.searchsorted(et,np.asarray(t),side='right')]
 volumes[lab]=volume
 if lab in TRAIN:b=replace(b,pulses={**b.pulses,'N':tuple((t,d*2/float(volume(t))) for t,d in b.pulses['N'])})
 if lab=='LAB017':b=replace(b,pulses={**b.pulses,'N':((min(t for t,d in b.pulses['N']),sum(d for t,d in b.pulses['N'])),)})
 if lab=='LAB021':b=replace(b,initials={**b.initials,'X':9.*.03})
 register_temperature(lab,v.raw_temperature[lab])
 batches[lab]=b;cores[lab]=simulate(b,theta)
 entry.append(dict(LAB=lab,**b.initials,pulsos=b.pulses['N'],corte_h=v.horizons[lab]))
 obs[lab]=obs[lab].loc[obs[lab].time_h.between(0,v.horizons[lab])].copy()
display(pd.DataFrame(entry))

from types import FunctionType

def gas_inputs(frames, batch_inputs, chemical_parameters, *, horizon=None):
 source={}; active={}
 for lab,trajectory in frames.items():
  frame=with_jumps(trajectory)
  batch=batch_inputs[lab]
  begin=propagation_start(batch)
  end=float(frame.index[-1]) if horizon is None else horizon
  available,_=thermal_profiles[lab]
  times=np.unique(np.round(np.r_[np.arange(begin,end,.1),end,available[(available>begin)&(available<end)]],12))
  assert np.min(np.diff(times))>1e-10
  midpoints=(times[:-1]+times[1:])/2
  interval_temperatures=causal_temperature(batch,times[:-1])
  rates=[0.]
  p=parameter_vector(chemical_parameters)
  for time,temperature in zip(midpoints,interval_temperatures):
   y=np.array([np.interp(time,frame.index,frame[state]) for state in STATES])
   mu,beta,_,_=sugar_terms(y,p,temperature)
   rates.append((mu/12.72+beta/1.05)*y[0])
  source[lab]=(times,np.asarray(rates))
  initial_sugar=batch.initials['G']+batch.initials['F']
  consumed=initial_sugar-np.interp(np.r_[begin,midpoints],frame.index,frame.S)
  z=np.clip((consumed-5.)/5.,0.,1.)
  active[lab]=z*z*(3.-2.*z)
  assert np.all(active[lab][consumed<=5.]==0) and np.all(active[lab][consumed>=10.]==1)
 return source,active

PCO2_BAR=1.0
ETHANOL_G_L_PER_VOL_PERCENT=7.8924

def saturation(T,E,S):
 a=np.maximum(E,0)/ETHANOL_G_L_PER_VOL_PERCENT
 return 3.295*np.exp((7e-5*a-.02905)*np.asarray(T)-.0179*a-.00111*np.maximum(S,0))*PCO2_BAR

def configure_gas(frames):
 global gas_equilibrium
 gas_equilibrium={}
 for lab,f in frames.items():
  times=caches[lab][0]
  ff=with_jumps(f)
  batch=batches[lab]
  T=causal_temperature(batch,times)
  E=np.interp(times,ff.index,ff.E)
  S=np.interp(times,ff.index,ff.S)
  gas_equilibrium[lab]=(T,E,S,saturation(T,E,S))

def gas(lab,k,gain=1,activated=True,substeps=1):
 t,source=caches[lab]
 source=source*activation[lab] if activated else source.copy()
 T,E,S,sat=gas_equilibrium[lab]
 cd=np.zeros(len(t));qout=np.zeros(len(t));acc=np.zeros(len(t))
 qout[0]=max(source[0]-k*sat[0],0.)
 for i,dt in enumerate(np.diff(t),1):
  qp=source[i];previous=cd[i-1];current=previous
  for j in range(substeps):
   h=dt/substeps;fraction=(j+.5)/substeps;cs=saturation(T[i-1],E[i-1]*(1-fraction)+E[i]*fraction,S[i-1]*(1-fraction)+S[i]*fraction)
   if current>cs:
    current=cs+(current-cs)*np.exp(-k*h)
   else:
    D=cs-current
    if qp<=0 or D<=0:delta=0.
    elif k*D<=qp:delta=D*(-np.expm1(-k*h))
    else:
     crossing=(D-qp/k)/qp;linear=min(h,crossing);delta=qp*linear
     if crossing<h:delta+=(qp/k)*(-np.expm1(-k*(h-crossing)))
    current+=delta
  cd[i]=current;acc[i]=(current-previous)/dt;qout[i]=qp-acc[i]
 assert cd.min()>=-1e-12 and qout.min()>-1e-10, (lab,cd.min(),qout.min())
 assert np.max(abs(np.diff(cd)-np.diff(t)*(source[1:]-qout[1:])))<1e-10
 return pd.DataFrame(dict(qprod=source,Cd=cd,Csat=sat,T=T,E=E,S=S,qgas=qout,qpred=gain*qout),index=t)

KLA,GAIN=(0.13796553316180868, 0.6586783889606775)
reference_cores={label: {lab: simulate(batch, p) for lab,batch in batches.items()}
                 for label,p in parameters.items() if label != causal_label}

def gas_curves(frames, batch_inputs, chemical_parameters, *, horizon=None):
 source,active=gas_inputs(frames,batch_inputs,chemical_parameters,horizon=horizon)
 context=dict(globals(),batches=batch_inputs,caches=source,activation=active)
 FunctionType(configure_gas.__code__,context)(frames)
 predict=FunctionType(gas.__code__,context,argdefs=gas.__defaults__)
 return {lab: predict(lab,KLA,GAIN) for lab in frames}


caches,activation=gas_inputs(cores,batches,theta)
curves=gas_curves(cores,batches,theta)
reference_gas={label: gas_curves(frames,batches,parameters[label])
               for label,frames in reference_cores.items()}
metrics=[]
for lab in LABS:
 o=obs[lab];f=curves[lab];e=np.interp(o.time_h,f.index,f.qpred)-o.co2_raw_g_l_h
 metrics.append(dict(LAB=lab,rol='Lotes usados para ajustar gas' if lab in TRAIN else 'Holdout gas',n=len(o),RMSE=np.sqrt(np.mean(e*e)),sesgo=np.mean(e)))
metrics=pd.DataFrame(metrics);display(metrics);display(metrics.groupby('rol')[['RMSE','sesgo']].mean())

display(pd.DataFrame([dict(LAB=lab,inicio_5g_h=float(caches[lab][0][np.flatnonzero(activation[lab]>0)[0]]),completa_10g_h=float(caches[lab][0][np.flatnonzero(activation[lab]>=1)[0]])) for lab in LABS]))

## Holdout LAB022–024

Predicción de ambos ajustes del modelo S con temperatura causal y los coeficientes de gas del notebook 17. La simulación abarca hasta las 190 h. Las condiciones iniciales siguen las del notebook 13. La química de inoculación y la primera muestra de biomasa a las 0,25 h inicializan la simulación. Para el etanol inicial se usan 10,010194 g/L. La temperatura proviene del respaldo del ensayo.

En la hora registrada de nutrición se aplica un pulso de 1 g de SFX y 0,4 g de FDA por lote. Con el volumen remanente y la conversión del notebook 13, el aporte nominal es 99,9 mg N/L. Las muestras posteriores se usan para evaluar las predicciones en sus horas de toma. El flujo se compara en ventanas de 30 minutos con observaciones reales válidas. Las curvas de etanol y CO₂ disuelto muestran las predicciones disponibles para esas variables.

El crecimiento se detiene cuando se agota el azúcar total. Se usa el mismo integrador químico para todos los lotes.

La última fila de las figuras corresponde a LAB022–024. Sus paneles de flujo muestran promedios de 30 minutos. Las filas previas usan el soporte de medición de sus ensayos.

In [ ]:
import nbformat

instrument = json.loads((ROOT / "configs/co2_instrument.json").read_text(encoding="utf-8"))
prep_path = ROOT / "notebooks/preparing_data/lab022_024_data.source.ipynb"
prep_nb = nbformat.read(prep_path, as_version=4)
sample_code = [cell.source for cell in prep_nb.cells if cell.cell_type == "code" and
               any(marker in cell.source for marker in
                   ("workbook = data_dir /", "y15_sources = {", "oculyze_frames = []"))]
assert len(sample_code) == 3
prepared = {}
for source in sample_code:
    python_source = "\n".join(line for line in source.splitlines() if not line.startswith("%"))
    exec(compile(python_source, str(prep_path), "exec"), prepared)

holdout_labs = prepared["labs"]
holdout_biomass = prepared["oculyze"].assign(X_g_L=lambda f: .03 * f["Viable concentration"])
holdout_chemistry = prepared["chemistry"]
holdout_nutrition = (prepared["nutrition"] - prepared["inoculation"]).dt.total_seconds() / 3600
holdout_horizon = 190.
# Dosis confirmada por lote. Equivalente de YAN de SFX y fracción de N en FDA.
pulse_mass_n_g = 1.0 * .10 + .4 * (2 * 14.007 / 132.1)

reactors = prepared["experiments"]["Fermentador"]
reactor_map = pd.Series(reactors.index, index=reactors)
def read_holdout_backup(filename, value, *, include_before_start=False):
    frame = pd.read_csv(prepared["data_dir"] / filename,
                        usecols=["timestamp", "fermentador", value], parse_dates=["timestamp"])
    frame["lab"] = frame["fermentador"].map(reactor_map)
    frame["h"] = (frame["timestamp"] - frame["lab"].map(prepared["inoculation"])).dt.total_seconds() / 3600
    return (frame if include_before_start else frame.loc[frame["h"].ge(0)]).sort_values("h")

holdout_temperature = read_holdout_backup("backup_global_temperatura.csv", "T", include_before_start=True)
holdout_raw_flow = read_holdout_backup("backup_global_co2.csv", "flow_sccm")
mass_factor = 44.0095 * 60 * instrument["co2_correction_factor"] / (1000 * 24.16)
holdout_batches, holdout_flow, pulse_rows = {}, {}, []
for lab in holdout_labs:
    samples = prepared["offline"].loc[prepared["offline"]["lab"].eq(lab)]
    withdrawals = samples.loc[samples["Observaciones"].eq("se quitan 50ml"), "hours"].to_numpy()
    initial_volume = float(prepared["experiments"].at[lab, "Volumen_Inicial_L"].removesuffix("L"))
    pulse_h = holdout_nutrition[lab]
    pulse_volume = initial_volume - .05 * np.searchsorted(withdrawals, pulse_h, side="right")
    pulse_n = pulse_mass_n_g / pulse_volume
    start_sample = holdout_biomass.loc[holdout_biomass["lab"].eq(lab) &
                                       holdout_biomass["sample"].eq("01")].iloc[0]
    holdout_start = float(start_sample["hours"])
    assert holdout_start == .25
    chem = holdout_chemistry.loc[holdout_chemistry["lab"].eq(lab)]
    initial = chem.loc[chem["sample"].eq("01")].iloc[0]
    initials = {"X": start_sample["X_g_L"], "N": initial["YAN for plot"] / 1000,
                "G": initial["GLUCOSE-320"], "F": initial["Fructose (derived)"], "E": 10.010194}
    biomass_h = holdout_biomass.loc[holdout_biomass["lab"].eq(lab), "hours"]
    # Se incluyen las horas de muestreo para evaluar la solución directamente.
    times = np.unique(np.r_[np.arange(holdout_start, holdout_horizon, .1), holdout_horizon, pulse_h,
                           chem.loc[chem["hours"].ge(holdout_start), "hours"], biomass_h])
    temp = holdout_temperature.loc[holdout_temperature.lab.eq(lab)].rename(columns={'h':'t_h'})
    register_temperature(lab,temp)
    holdout_batches[lab] = kin.BatchData(
        medium="natural", batch=lab, time=times, temperature_c=np.full(len(times),np.nan),
        pulses={"N": ((pulse_h, pulse_n),)}, observations={}, initials=initials,
    )
    pulse_rows.append({"LAB": lab, "Pulso [h]": pulse_h, "Volumen [L]": pulse_volume,
                       "Aporte [mg N/L]": 1000 * pulse_n})
    raw = holdout_raw_flow.loc[holdout_raw_flow["lab"].eq(lab) &
                               holdout_raw_flow["h"].gt(0)].copy()
    assert raw["h"].max() >= holdout_horizon
    raw["bin"] = np.ceil(raw["h"] / .5 - 1e-10).astype(int)
    holdout_sample_volumes = initial_volume - .05 * np.searchsorted(withdrawals, raw["h"], side="right")
    raw["mass_flow"] = raw["flow_sccm"] * mass_factor / holdout_sample_volumes
    means = {}
    for index, window in raw.groupby("bin"):
        end = index * .5
        if end > holdout_horizon:
            break
        gaps = np.diff(np.r_[end - .5, window["h"], end])
        valid = len(window) >= 30 and gaps.max() <= 2 / 60 + 1e-10
        means[end] = window["mass_flow"].mean() if valid else np.nan
    holdout_flow[lab] = pd.Series(means).reindex(np.arange(.5, holdout_horizon + .5, .5))
display(pd.DataFrame(pulse_rows).set_index("LAB").round(3))

In [ ]:
holdout_kinetic = {lab: simulate(batch,theta) for lab,batch in holdout_batches.items()}
reference_holdout = {label: {lab: simulate(batch,p) for lab,batch in holdout_batches.items()}
                     for label,p in parameters.items() if label != causal_label}
for frames in [holdout_kinetic,*reference_holdout.values()]:
    for lab,frame in frames.items():
        [jump] = frame.attrs["pulse_jumps"]
        np.testing.assert_allclose([jump["t"], jump["after"][1]-jump["before"][1]],
                                   holdout_batches[lab].pulses["N"][0], atol=1e-12)

holdout_gas = gas_curves(holdout_kinetic,holdout_batches,theta,horizon=holdout_horizon)
reference_holdout_gas = {label: gas_curves(frames,holdout_batches,parameters[label],horizon=holdout_horizon)
                         for label,frames in reference_holdout.items()}

def gas_window_means(frames):
    means = {}
    for lab,curve in frames.items():
        times = curve.index.to_numpy()
        cumulative = np.r_[0.,np.cumsum(np.diff(times)*curve["qpred"].to_numpy()[1:])]
        ends = holdout_flow[lab].index.to_numpy()
        ends = ends[ends-.5 >= times[0]]
        means[lab] = pd.Series((np.interp(ends,times,cumulative) -
                                np.interp(ends-.5,times,cumulative))/.5, index=ends)
    return means

holdout_mean_gas = gas_window_means(holdout_gas)
reference_mean_gas = {label: gas_window_means(frames) for label,frames in reference_holdout_gas.items()}

## Temperatura · LAB016–024

Las medianas del minuto aparecen en la hora de cierre. La línea escalonada muestra el valor disponible para el modelo.

In [ ]:
fig,axes=plt.subplots(3,3,figsize=(15,12))
for ax,lab in zip(axes.flat,[*LABS,*holdout_labs]):
 batch=(batches if lab in LABS else holdout_batches)[lab]
 temperature_panel(ax,batch,' · gas calibration' if lab in TRAIN else ' · HOLDOUT')
fig.suptitle('Temperature · LAB016–024',y=.99)
fig.tight_layout(pad=.7,h_pad=1.2,w_pad=.9)
plt.show()

temperature_audit=[]
for batch in [*treated,*batches.values(),*holdout_batches.values()]:
 available,values=thermal_profiles[batch.batch]
 first,end=batch.time[[0,-1]]
 begin=propagation_start(batch)
 updates=available[(available>begin)&(available<=end)]
 assert len(updates)>0 and np.isfinite(causal_temperature(batch,updates)).all()
 # Compare against the online lookup immediately before and after window closure.
 probes=updates[::max(1,len(updates)//20)]
 positions=np.searchsorted(available,probes)
 np.testing.assert_array_equal(causal_temperature(batch,probes),values[positions])
 np.testing.assert_array_equal(causal_temperature(batch,probes-1e-6),values[positions-1])
 temperature_audit.append(dict(LAB=batch.batch,espera_min=(begin-first)*60,
    temperatura_inicial_C=causal_temperature(batch,begin),
    minutos_con_datos=len(updates),
    max_retencion_min=np.diff(np.r_[begin,updates,end]).max()*60))
display(pd.DataFrame(temperature_audit).set_index('LAB').round(3))
print('TEMPERATURE_AUDIT_JSON',json.dumps(temperature_audit))

In [ ]:
panels = {
    'X': ('Biomass','g/L',holdout_biomass,'X_g_L',1.),
    'N': ('YAN','mg/L',holdout_chemistry,'YAN for plot',1000.),
    'S': ('Total sugar','g/L',holdout_chemistry,'GLU-FRU-320',1.),
    'E': ('Ethanol','g/L',None,None,1.),
}
gas_unit = 'g/(L h)'


def draw_recent(ax,lab,state):
    group = 'LAB022–024'
    if state=='CO2':
        traces = chemical_frames(lab,holdout_mean_gas,reference_mean_gas)
        observed = holdout_flow[lab].dropna()
        observed = observed.loc[observed.index.intersection(traces[causal_label].index)]
        ax.plot(observed.index,observed,color='black',lw=1,label='Samples')
        predictions = {label: trace.loc[observed.index].to_numpy() for label,trace in traces.items()}
        ax._rmse_text = score_comparison(group,lab,state,gas_unit,observed.index,observed.to_numpy(),predictions)
        title,unit = 'CO₂',gas_unit
    elif state=='Cd':
        frames = chemical_frames(lab,holdout_gas,reference_holdout_gas)
        traces = {label: frame.Cd for label,frame in frames.items()}
        title,unit = 'Dissolved CO₂','g/L'
    else:
        title,unit,sample_frame,column,scale = panels[state]
        frames = chemical_frames(lab,holdout_kinetic,reference_holdout)
        traces = {label: with_jumps(frame)[state]*scale for label,frame in frames.items()}
        if sample_frame is not None:
            samples = sample_frame.loc[sample_frame.lab.eq(lab)].dropna(subset=[column])
            ax.scatter(samples.hours,samples[column],c='black',s=23,label='Samples',zorder=5)
            evaluation = samples.loc[samples['sample'].ne('01')]
            times = evaluation.hours.to_numpy()
            predictions = {label: frame.loc[times,state].to_numpy()*scale for label,frame in frames.items()}
            ax._rmse_text = score_comparison(group,lab,state,unit,times,evaluation[column].to_numpy(),predictions)
        else:
            ax.text(.97,.9,'No post-initial samples',ha='right',transform=ax.transAxes,fontsize=9)
        if state=='S':
            density = prepared['offline']
            density = density.loc[density.lab.eq(lab)&density.hours.between(0,holdout_horizon)].dropna(subset=['Densidad_g_L'])
            ax.scatter(density.hours,2.5*(density.Densidad_g_L-1000),marker='D',facecolors='none',
                       edgecolors='#D55E00',s=32,label='G+F from density',zorder=4)
            ax.axhline(0,color='gray',lw=.6)
    plot_comparison(ax,traces,holdout_horizon)
    ax.axvline(holdout_nutrition[lab],c='gray',ls=':',lw=.8,label='N pulse')
    ax.set(title=f'{lab} · HOLDOUT',xlabel='Time [h]',ylabel=f'{title} [{unit}]',
           xlim=(-2,holdout_horizon+2),xticks=[0,25,50,75,100,125,150,175,190])
    ax.grid(alpha=.2)


def finish_panels(fig,axes,title,state):
    for ax,lab in zip(axes[-1],holdout_labs):
        draw_recent(ax,lab,state)
    for ax in axes.flat:
        lower,upper=ax.get_ylim()
        y_min,y_max=ax.dataLim.intervaly
        padding=.05*(y_max-y_min)
        if y_min<lower or y_max>upper:
            ax.set_ylim(min(lower,y_min-padding),max(upper,y_max+padding))
        lower,upper=ax.get_ylim()
        for line in ax.lines:
            if line.get_label() in parameters:
                values=np.asarray(line.get_ydata(),float)
                assert np.isfinite(values).all() and values.min()>=lower and values.max()<=upper
        legend_inside(ax)
    fig.suptitle(title,y=.99)
    fig.tight_layout(pad=.7,h_pad=1.2,w_pad=.9)
    plt.show()


fig,axes=plt.subplots(3,3,figsize=(15,12))
for ax,lab in zip(axes[:2].flat,LABS):
    frames=chemical_frames(lab,curves,reference_gas)
    traces={label: frame.qpred for label,frame in frames.items()}
    observed=obs[lab]
    times=observed.time_h.to_numpy()
    predictions={label: np.interp(times,trace.index,trace) for label,trace in traces.items()}
    group='LAB019–021' if lab in TRAIN else 'LAB016–018'
    ax._rmse_text=score_comparison(group,lab,'CO2',gas_unit,times,observed.co2_raw_g_l_h.to_numpy(),predictions)
    ax.plot(times,observed.co2_raw_g_l_h,c='black',lw=1,label='Samples')
    plot_comparison(ax,traces,v.horizons[lab])
    ax.set(title=lab+(' · gas calibration' if lab in TRAIN else ' · HOLDOUT'),
           xlabel='Time [h]',ylabel='CO₂ [g/(L·h)]',ylim=(0,None))
    ax.grid(alpha=.2)
finish_panels(fig,axes,'CO₂ · LAB016–024','CO2')

for state,(title,unit,_,_,scale) in panels.items():
    fig,axes=plt.subplots(3,3,figsize=(15,12))
    for ax,lab in zip(axes[:2].flat,LABS):
        frames=chemical_frames(lab,cores,reference_cores)
        traces={label: with_jumps(frame)[state]*scale for label,frame in frames.items()}
        times=np.array([])
        observed=np.array([])
        sample_label='Samples'
        if lab in TRAIN:
            assay='GF' if state=='S' else state
            rows=chemical_observations(v.chem,lab,assay)
            times=rows.t_h.to_numpy()
            observed=rows[CHEMICAL_COLUMNS[assay]].to_numpy()*scale
        elif state=='X':
            rows=v.biomass.loc[v.biomass.lab.eq(lab)].dropna(subset=['t_h','X_viable_g_l'])
            times=rows.t_h.to_numpy()
            observed=rows.X_viable_g_l.to_numpy()
        elif state=='S':
            rows=v.density.loc[v.density.lab.eq(lab)].dropna(subset=['t_h','density_g_l'])
            times=rows.t_h.to_numpy()
            observed=2.5*(rows.density_g_l.to_numpy()-1000)
            sample_label='G+F from density'
        score=(times>0)&(times<=v.horizons[lab])
        predictions={label: np.interp(times[score],trace.index,trace) for label,trace in traces.items()}
        group='LAB019–021' if lab in TRAIN else 'LAB016–018'
        ax._rmse_text=score_comparison(group,lab,state,unit,times[score],observed[score],predictions)
        if not score.any():
            ax.text(.97,.9,'No post-initial samples',ha='right',transform=ax.transAxes,fontsize=9)
        elif sample_label=='G+F from density':
            ax.scatter(times[score],observed[score],marker='D',facecolors='none',edgecolors='#D55E00',s=32,label=sample_label)
        else:
            ax.scatter(times[score],observed[score],c='black',s=23,label=sample_label,zorder=5)
        if state!='S' or lab in TRAIN:
            initial=batches[lab].initials
            value=initial['G']+initial['F'] if state=='S' else initial[state]
            ax.scatter([0.],[value*scale],c='black',s=23,label='Samples')
        if state=='S' and lab in TRAIN:
            density=v.density.loc[v.density.lab.eq(lab)].dropna(subset=['t_h','density_g_l'])
            ax.scatter(density.t_h,2.5*(density.density_g_l-1000),marker='D',facecolors='none',
                       edgecolors='#D55E00',s=32,label='G+F from density')
        if state=='N':
            for time,dose in batches[lab].pulses['N']:
                ax.axvline(time,c='gray',ls=':',lw=.8,label='N pulse')
        if state=='S':
            ax.axhline(0,color='gray',lw=.6)
        plot_comparison(ax,traces,v.horizons[lab])
        if state!='S':
            ax.set_ylim(bottom=0)
        ax.set(title=lab,xlabel='Time [h]',ylabel=f'{title} [{unit}]')
        ax.grid(alpha=.2)
    finish_panels(fig,axes,title+' · Holdout',state)

fig,axes=plt.subplots(3,3,figsize=(15,12))
for ax,lab in zip(axes[:2].flat,LABS):
    frames=chemical_frames(lab,curves,reference_gas)
    plot_comparison(ax,{label: frame.Cd for label,frame in frames.items()},v.horizons[lab])
    ax.set(title=lab,xlabel='Time [h]',ylabel='Dissolved CO₂ [g/L]')
    ax.grid(alpha=.2)
finish_panels(fig,axes,'Dissolved CO₂ · model predictions','Cd')

## Comprobaciones de integración y balance del depósito

La tabla verifica por separado los dos ajustes en LAB016–021. El depósito cierra sobre la fuente efectiva de CO₂ antes de aplicar la ganancia. Se incluyen las retiradas de líquido y se comprueba la sensibilidad de la integración química al reducir el paso a la mitad.

In [ ]:
display(pd.DataFrame([dict(LAB=lab,t_h=j['t'],dosis_g_L=j['dose'],N_pre_g_L=j['before'][1],N_post_g_L=j['after'][1]) for lab,f in {**traincurves,**cores}.items() for j in f.attrs['pulse_jumps']]))
display(pd.DataFrame([dict(LAB=lab,**f.attrs['minimum_integrated_states']) for lab,f in cores.items()]))
masschecks=[]
for label,frames in {**reference_gas,causal_label:curves}.items():
 for lab,f in frames.items():
    t=f.index.to_numpy();V=volumes[lab](t);dt=np.diff(t)
    q=f.qprod.to_numpy();cd=f.Cd.to_numpy();out=f.qgas.to_numpy()
    produced=np.sum(V[:-1]*dt*q[1:]);emitted=np.sum(V[:-1]*dt*out[1:])
    removed=np.sum((V[:-1]-V[1:])*cd[1:])
    err=V[-1]*cd[-1]-V[0]*cd[0]-produced+emitted+removed
    assert abs(err)<1e-8
    masschecks.append(dict(ajuste=label,LAB=lab,produccion_g=produced,gas_g=emitted,
                           retirado_disuelto_g=removed,final_disuelto_g=V[-1]*cd[-1],error_g=err))
display(pd.DataFrame(masschecks))
assert all(hashlib.sha256(p.read_bytes()).hexdigest()==h for p,h in hashes.items())
print('Archivos previos intactos; ninguna configuración ni dato sobrescrito.')
assert all(theta[k]==v for k,v in fixed.items())
print('Coeficientes de gas del modelo S del notebook 17:',KLA,GAIN)
print('RMSE medio gas por rol:',metrics.groupby('rol').RMSE.mean().to_dict())

# Evaluación independiente de términos_s, extraída del notebook 17.
reference_path=ROOT/'notebooks/17_zenteno_single_sugar_variable_solubility.source.ipynb'
reference_nb=nbformat.read(reference_path,4)
reference_code=next(cell.source for cell in reference_nb.cells if 'def terms_s(' in cell.source)
reference_terms=reference_code[reference_code.index('def terms_s'):reference_code.index('def simulate_s')]
ref_scope=dict(np=np,kin=kin,theta=theta)
exec(compile(reference_terms,str(reference_path),'exec'),ref_scope)
for state in (np.array([1.5,.07,125.,45.]),np.array([2.,.08,0.,95.])):
    for temperature in (16.,21.,30.):
        batch=replace(treated[0],temperature_c=np.full(len(treated[0].time),temperature))
        mu,beta,kd,maintenance=ref_scope['terms_s'](0.,state,theta,batch)
        expected=state[0]*np.array([mu-kd,-mu/theta['Yxn'],
                                    -(mu/theta['Yxs']+beta/theta['Yes']+maintenance),beta])
        np.testing.assert_allclose(chemical_rhs(state,parameter_vector(theta),temperature),expected,rtol=1e-12,atol=1e-12)

step_checks=[]
for label,p in parameters.items():
    for batch in [*treated,*batches.values(),*holdout_batches.values()]:
        step_error=(simulate(batch,p)-simulate(batch,p,max_step=1/120)).abs().max()
        step_checks.append(dict(ajuste=label,LAB=batch.batch,**step_error.to_dict()))
step_checks=pd.DataFrame(step_checks).set_index(['ajuste','LAB'])
display(step_checks.round(5))
print('RK4_HALF_STEP_MAX_JSON',step_checks.max().to_json())

## Pico de CO₂ por lote

El pico es el máximo de la señal de flujo en las horas con datos válidos. En LAB016–021 se usa el soporte horario de la señal observada y se evalúa la predicción en esas mismas horas. En LAB022–024 se comparan medias de 30 minutos sobre ventanas comunes. La hora se mide desde el origen del ensayo. Un error de tiempo positivo indica un pico tardío y uno de altura positivo indica un pico sobreestimado.

In [ ]:
peak_rows=[]
for lab in [*LABS,*holdout_labs]:
    if lab in LABS:
        observed=obs[lab].set_index('time_h').co2_raw_g_l_h.dropna()
        frames=chemical_frames(lab,curves,reference_gas)
        traces={label: pd.Series(np.interp(observed.index,frame.index,frame.qpred),
                                 index=observed.index)
                for label,frame in frames.items()}
    else:
        traces=chemical_frames(lab,holdout_mean_gas,reference_mean_gas)
        observed=holdout_flow[lab].dropna()
        observed=observed.loc[observed.index.intersection(traces[causal_label].index)]
        traces={label: trace.loc[observed.index] for label,trace in traces.items()}

    observed_height=float(observed.max())
    observed_time=float(observed.idxmax())
    for label,trace in traces.items():
        assert trace.index.equals(observed.index)
        assert np.isfinite(trace).all()
        height=float(trace.max())
        time=float(trace.idxmax())
        peak_rows.append(dict(LAB=lab,ajuste=label,pico_observado=observed_height,
                              hora_observada=observed_time,pico_predicho=height,
                              hora_predicha=time,error_altura=height-observed_height,
                              error_hora=time-observed_time))

peaks=pd.DataFrame(peak_rows)
display(peaks.round(4))
peak_errors=(peaks.assign(MAE_altura=peaks.error_altura.abs(), MAE_hora=peaks.error_hora.abs())
             .groupby('ajuste')[['MAE_altura','MAE_hora']].mean()
             .reindex(list(parameters)))
display(peak_errors.round(4))
print('CO2_PEAKS_JSON',peaks.to_json(orient='records'))
print('CO2_PEAK_MAE_JSON',peak_errors.to_json(orient='index'))

## Comparación de RMSE con temperatura causal

Cada par de RMSE se calcula sobre las mismas observaciones. Se promedian los lotes con igual peso dentro de cada grupo. En LAB016–018 el azúcar total se infiere por densidad. En LAB022–024 se contrasta con GLU-FRU-320 y el etanol posterior al inicio se muestra como predicción.

In [ ]:
paired=pd.DataFrame(paired_scores)
keys=['grupo','LAB','variable','unidad']
assert paired.groupby(keys).ajuste.nunique().eq(len(parameters)).all()
assert paired.groupby(keys).n.nunique().eq(1).all()
by_lab=paired.pivot(index=keys,columns='ajuste',values='RMSE').reindex(columns=list(parameters))
by_lab['Δ causal − offline']=by_lab[causal_label]-by_lab[offline_label]
display(by_lab.round(4))
summary=by_lab.groupby(level=['grupo','variable','unidad']).mean()
display(summary.round(4))
print('PAIRED_RMSE_JSON',summary.reset_index().to_json(orient='records'))

## Cambios de parámetros al ajustar con temperatura causal

La columna del notebook 17 sirve como referencia de la calibración publicada. Los porcentajes comparan exclusivamente los dos reajustes con el mismo peso de biomasa y el mismo tratamiento de inicio.

In [ ]:
parameter_audit=pd.DataFrame({
    'Notebook 17': pd.Series(published_theta),
    offline_label: pd.Series(parameters[offline_label]),
    causal_label: pd.Series(parameters[causal_label]),
})
parameter_audit['Δ %']=100*(parameter_audit[causal_label]/parameter_audit[offline_label]-1)
parameter_audit['Libre']=parameter_audit.index.isin(names)
parameter_audit['Mínimo']=pd.Series({k: v[0] for k,v in bounds.items()})
parameter_audit['Máximo']=pd.Series({k: v[1] for k,v in bounds.items()})
parameter_audit['Distancia al máximo (%)']=100*(1-parameter_audit[causal_label]/parameter_audit['Máximo'])
ordered=parameter_audit.loc[names].sort_values('Δ %',key=abs,ascending=False)
assert parameter_audit.loc[~parameter_audit.Libre,'Δ %'].eq(0).all()
display(ordered.round(6))
display(parameter_audit.loc[~parameter_audit.Libre,['Notebook 17',offline_label,causal_label]].round(8))
print('PARAMETER_AUDIT_JSON',ordered.to_json(orient='index'))

## Parámetros archivados para EKF 14

Este registro conserva los vectores de la comparación anterior de ponderaciones de biomasa que carga EKF 14. Las figuras y tablas de temperatura de este notebook usan los dos ajustes nuevos.

In [ ]:
archived_biomass_fits = json.loads('{"Par\\u00e1metros S del 17": {"mu0": 0.09085058816701197, "betaS0": 0.8189664538991809, "Ks0": 23.046564567082594, "Kie0": 96.26516563486751, "Yxn": 6.497331613020979, "Yes": 0.4893180893478443, "Kn0": 0.008257860224319971, "Kd0": 0.00044, "m0": 0.0183385793928771, "Yxs": 0.5165070815658868}, "Refit \\u03c3X = 0.50": {"Kie0": 96.26516563486751, "Kn0": 0.008257860224319971, "Kd0": 0.00044, "m0": 0.0183385793928771, "Yxs": 0.5165070815658868, "mu0": 0.09103862171169734, "betaS0": 0.8582516117752647, "Ks0": 26.39905602592316, "Yxn": 6.476857639199847, "Yes": 0.4893936377526879}, "Refit \\u03c3X = 0.25": {"Kie0": 96.26516563486751, "Kn0": 0.008257860224319971, "Kd0": 0.00044, "m0": 0.0183385793928771, "Yxs": 0.5165070815658868, "mu0": 0.08729282025853075, "betaS0": 0.908349170132679, "Ks0": 23.09250367828602, "Yxn": 5.720952281355235, "Yes": 0.4864193257781055}}')
print('CHEMICAL_PARAMETERS_JSON',json.dumps(archived_biomass_fits))